# Tunix-Med: High-Performance SFT with JAX/Tunix

This notebook fine-tunes `google/gemma-3-1b-it` on the Medical QA dataset using **Tunix**, a JAX-native library for large-scale post-training. 

### Why JAX/Tunix?
- **Hardware Agnostic**: Runs seamlessly on TPU, NVIDIA GPU (via XLA), and Apple Silicon (via MPS).
- **Scalability**: Designed for multi-host training and optimized with XLA compilation.
- **Efficiency**: Leverages high-performance model implementations like MaxText.

### What this notebook does
1. Sets up the JAX environment and verifies device acceleration.
2. Prepares the dataset for Tunix's JAX-based trainer.
3. Performs Supervised Fine-Tuning (SFT) with LoRA.
4. Exports the fine-tuned adapter for inference.

In [ ]:
import jax
from jax.lib import xla_bridge

# Check JAX device - This will work for TPU, GPU, and MPS
device_backend = xla_bridge.get_backend().platform
print(f"JAX is using backend: {device_backend}")
print(f"Available devices: {jax.devices()}")

if device_backend == "cpu":
    print("Warning: JAX is running on CPU. Performance will be limited.")

## Installation of Tunix and JAX dependencies

In [ ]:
%%capture
!pip install --upgrade jax jaxlib
!pip install git+https://github.com/google/tunix.git
!pip install datasets transformers huggingface_hub

## Fine-Tuning with Tunix

Tunix provides a structured way to perform SFT. We will configure the training job using the Tunix API or its CLI-style runner.

In [ ]:
# Example Tunix SFT Configuration
sft_config = {
    "model_name": "google/gemma-3-1b-it",
    "dataset_name": "lmassaron/Medical_QA_ft",  # Assume we pushed the medical data here
    "per_device_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_epochs": 1,
    "lora": {"rank": 8, "alpha": 32, "target_modules": "all-linear"},
    "output_dir": "tunix-medical-model",
}

print("Starting Tunix SFT Training...")
# sft.train(sft_config) # This would execute the training

## Exporting the Model
After training, Tunix saves the adapter weights which can be loaded back into Transformers or used for inference with vLLM.